In [1]:
# install packages in colab if necessary
#!pip install nlpaug tdqm sacremoses

import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.sentence as nas
import nlpaug.flow as nafc
from transformers import MarianMTModel, MarianTokenizer
import numpy as np
import pandas as pd
import torch
import gc
from tqdm import tqdm
import re

tqdm.pandas()


rseed = 42

In [ ]:
PLACEHOLDER = "MBTITOKENPLACEHOLDER"

def protect_mbti_mask(text):
    return text.replace("<mbti>", PLACEHOLDER)

def restore_mbti_mask(text):
    return re.sub(rf"{PLACEHOLDER}", "<mbti>", text, flags=re.IGNORECASE)


In [ ]:
# Synonym Augmenter
# prob 0.4
def synonymizer(text, aug1):
    protected = protect_mbti_mask(text)
#    coin = np.random.rand()
#    if coin < 0.4:
    augmented_text = aug1.augment(protected)
    augmented_text = augmented_text[0] if isinstance(augmented_text, list) else augmented_text
    return restore_mbti_mask(augmented_text)
#    else:
#        augmented_text = aug2.augment(text)
#        return augmented_text[0] if isinstance(augmented_text, list) else augmented_text

In [ ]:
# backtranslation
# prob 0.2
# def back_translation(text, aug):
#     augmented_text = aug.augment(text)
#     return augmented_text[0] if isinstance(augmented_text, list) else augmented_text
def backtranslate_safe(texts, device="cuda"):
    en_de_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
    en_de_mod = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-de").to(device)
    de_en_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-de-en")
    de_en_mod = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-de-en").to(device)

    gen_kwargs = dict(
        num_beams=4,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        max_length=512,
        early_stopping=True,
    )

    def translate(model, tokenizer, batch):
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                          truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, **gen_kwargs)
        return tokenizer.batch_decode(out, skip_special_tokens=True)

    protected = [protect_mbti_mask(t) for t in texts]
    results = []
    for i in range(0, len(protected), 16):
        batch = protected[i:i+16]
        de = translate(en_de_mod, en_de_tok, batch)
        back = translate(de_en_mod, de_en_tok, de)
        results.extend(back)

    del en_de_mod, de_en_mod
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return [restore_mbti_mask(t) for t in results]


# random swap
# prob 0.2
def random_swap(text, aug):
    augmented_text = aug.augment(text)
    return augmented_text[0] if isinstance(augmented_text, list) else augmented_text


#  random deletion
# prob 0.2
def random_del(text, aug):
    augmented_text = aug.augment(text)
    return augmented_text[0] if isinstance(augmented_text, list) else augmented_text


In [ ]:
# data local
#df = pd.read_csv("..\data\csv\mbti_cleaned.csv")

# data colab
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv("/content/drive/MyDrive/Data/mbti_cleaned.csv")

#data alvis
df = pd.read_csv("../data/mbti_cleaned.csv")

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Tim\AppData\Local\Temp\ipykernel_12148\1371491495.py:2: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_csv("..\data\csv\mbti_cleaned.csv")


In [ ]:
def label_samples(df, label, n_goal, rseed):
    obs_count = (df["label"] == label).sum()
    if obs_count < n_goal:
        df_subset = df[df["label"] == label]

        n_diff = n_goal - obs_count

        n_syn = round(n_diff*0.2)
        n_bt = round(n_diff*0.7)
        n_sw = round(n_diff*0.05)
        n_del = n_diff - (n_syn+n_bt+n_sw)



        syn_sample = df_subset.sample(n=n_syn, replace=True,random_state=rseed)
        bt_sample = df_subset.sample(n=n_bt, replace=True,random_state=rseed)
        sw_sample = df_subset.sample(n=n_sw, replace=True,random_state=rseed)
        del_sample = df_subset.sample(n=n_del, replace=True,random_state=rseed)

        return syn_sample, bt_sample, sw_sample, del_sample
    else:
        empty = df.iloc[:0]
        return empty, empty, empty, empty

    

    


In [24]:
syn_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])
bt_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])
sw_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])
del_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])

for label in df["label"].unique():
    s1, s2, s3, s4 = label_samples(df, label, n_goal=10000, rseed=42)
    

    syn_sample = pd.concat([syn_sample, s1]).dropna()
    bt_sample = pd.concat([bt_sample, s2]).dropna()
    sw_sample = pd.concat([sw_sample, s3]).dropna()
    del_sample = pd.concat([del_sample, s4]).dropna()


    



In [ ]:
def data_augmenter(syn_sample, bt_sample, sw_sample, del_sample):
    #device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    
    #aug_syn_word = naw.SynonymAug(aug_src = "wordnet")
    aug_syn_bert =naw.ContextualWordEmbsAug(
        model_path='bert-base-uncased',
        action="substitute",
        device="cuda"
    )
    # can also add wordnet if wanted
    syn_sample["post_augmented"] = syn_sample["post"].progress_apply(synonymizer, args=(aug_syn_bert,))

    # free up vram
    #del aug_syn_word
    del aug_syn_bert

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # aug_backtranslation = naw.BackTranslationAug(
    #     from_model_name="facebook/wmt19-en-de",
    #     to_model_name="facebook/wmt19-de-en",
    #     device= "cuda",
    #     batch_size=16
    # )

    texts = bt_sample["post"].tolist()
    bt_sample["post_augmented"] = backtranslate_safe(texts)

    # free up vram
    # del aug_backtranslation
    # gc.collect()

    # if torch.cuda.is_available():
    #     torch.cuda.empty_cache()


    aug_rand_swap = naw.RandomWordAug(action = "swap")
    sw_sample["post_augmented"] = sw_sample["post"].progress_apply(random_swap, args = (aug_rand_swap,))
    
    aug_rand_del = naw.RandomWordAug(action = "delete")
    del_sample["post_augmented"] = del_sample["post"].progress_apply(random_del, args = (aug_rand_del,))


    return syn_sample,bt_sample,sw_sample,del_sample



In [ ]:
# fixing nltk dependency
import nltk
#nltk.download('averaged_perceptron_tagger_eng')

# run data_augmenter
syn_df, bt_df, sw_df, del_df = data_augmenter(syn_sample, bt_sample, sw_sample, del_sample)

In [ ]:
#saving data
# syn_df.to_csv("syn_augmented.csv", sep='\t', index=False, quoting=1)
# bt_df.to_csv("bt_augmented.csv", sep='\t', index=False, quoting=1)
# sw_df.to_csv("sw_augmented.csv", sep='\t', index=False, quoting=1)
# del_df.to_csv("del_augmented.csv", sep='\t', index=False, quoting=1)
# print("Alle Dateien erfolgreich als CSV gespeichert!")

# reading finished datasets locally
syn_df = pd.read_csv("../data/csv/syn_augmented.csv", sep = "\t")
bt_df = pd.read_csv("../data/csv/bt_augmented.csv", sep = "\t")
sw_df = pd.read_csv("../data/csv/sw_augmented.csv", sep = "\t")
del_df = pd.read_csv("../data/csv/del_augmented.csv", sep = "\t")

bt_df.head()

,label,post,post_augmented
0,ENTJ,hey everyone. i just wanna get started with po...,"hey everybody. i just want to start posting, s..."
1,ENTJ,you know what really sucks? people being compl...,do you know what really sucks? people who are ...
2,ENTJ,"so why don't you just say hey bio, cut the typ...","So why don't you just say hey bio, cut me off ..."
3,ENTJ,"i second that. at first u will feel lonely, bu...","I second. first we will feel lonely, but then ..."
4,ENTJ,opinion: people who think money can't buy happ...,Opinion: People who think that money can't buy...
